In [2]:
import tensorflow as tf
from pathlib import Path

In [3]:
tf.random.set_seed(123)

project_dir = Path.cwd().parent
train_dir = project_dir / "img/train"
validation_dir = project_dir / "img/validate"

img_height = 180
img_width = 180
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

""" 
I created another notebook to give EfficientNetB0 architecture a try but there seems 
to be an issue with my versions of tf and keras as I tried many different tweaks and 
kept getting a shape mismatch error, so the MobileNetV2 architecture below seems 
like the best fit for my small dataset if I want to use transfer learning
"""
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

@tf.keras.utils.register_keras_serializable()
class MobileNetV2Preprocess(tf.keras.layers.Layer):
    def call(self, x):
        return tf.keras.applications.mobilenet_v2.preprocess_input(x)

model = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    MobileNetV2Preprocess(),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

model.save('../models/mobilenetv2_model.keras')

Found 161 files belonging to 3 classes.
Found 42 files belonging to 3 classes.


2026-04-06 04:07:57.000243: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-04-06 04:07:57.000382: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-06 04:07:57.000392: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-06 04:07:57.000605: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-06 04:07:57.000623: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/var/folders/04/s_rk8gfn6vq1608hxwwtl_040000gn/T/ipykernel_2803/2507554215.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weig

Epoch 1/20


2026-04-06 04:07:58.727355: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 525ms/step - accuracy: 0.3789 - loss: 1.7990 - val_accuracy: 0.4524 - val_loss: 1.0706
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.5590 - loss: 1.2000 - val_accuracy: 0.7143 - val_loss: 0.6899
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.6149 - loss: 1.0557 - val_accuracy: 0.7619 - val_loss: 0.5700
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.6149 - loss: 1.0595 - val_accuracy: 0.8095 - val_loss: 0.5113
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.6708 - loss: 1.0212 - val_accuracy: 0.7619 - val_loss: 0.4926
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.6770 - loss: 0.7843 - val_accuracy: 0.7619 - val_loss: 0.4435
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - accuracy: 0.7267 - loss: 0.8620 - val_accuracy: 0.8333 - val_loss: 0.4097
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - accuracy: 0.7019 - loss: 0.8476 - val_accuracy: 0.8095 - val_loss: 0.4041
Epoch 9/